# HBR YouTube OCR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/Data_crawling/blob/main/hbr_youtube_ocr/hbr_youtube_ocr_colab.ipynb?skip_cache=true)

Google Drive의 영상 경로와 로컬에서 만든 `roi.json` 경로 입력 → ROI 미리보기 → MangaOCR → 캐릭터명 매칭 → 시간축 병합 → 캐릭터별 TXT 저장.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install -U pytubefix manga-ocr opencv-python-headless
!pip -q install pandas==2.2.3
!pip -q install --force-reinstall --no-cache-dir pillow==11.3.0


In [ ]:
GDRIVE_VIDEO_PATH = "/content/drive/MyDrive/hbr.mp4" # @param {type:"string"}
GDRIVE_ROI_PATH = "/content/drive/MyDrive/roi.json" # @param {type:"string"}
OUTPUT_DIR = "/content/hbr_youtube_ocr" # @param {type:"string"}

from pathlib import Path
from google.colab import drive
import time

def stage(message):
    print(f"\n[STEP] {message}", flush=True)

stage("Google Drive 마운트")
drive.mount("/content/drive")

video_path = Path(GDRIVE_VIDEO_PATH)
roi_json_path = Path(GDRIVE_ROI_PATH)

if not video_path.exists():
    raise FileNotFoundError(video_path)

if not roi_json_path.exists():
    raise FileNotFoundError(roi_json_path)

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

print("video:", video_path)
print("roi:", roi_json_path)


In [ ]:
PREVIEW_TIME_SEC = 1635.0 # @param {type:"number"}

import cv2
import json
import matplotlib.pyplot as plt

roi_data = json.loads(
    roi_json_path.read_text(
        encoding="utf-8"
    )
)

cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps
cap.set(cv2.CAP_PROP_POS_MSEC, PREVIEW_TIME_SEC * 1000)
ok, frame = cap.read()
cap.release()

if not ok:
    raise RuntimeError("preview frame read failed")

def normalized_roi_to_box(roi, frame_width, frame_height):
    x1 = int(round(roi["x"] * frame_width))
    y1 = int(round(roi["y"] * frame_height))
    x2 = int(round((roi["x"] + roi["w"]) * frame_width))
    y2 = int(round((roi["y"] + roi["h"]) * frame_height))
    return x1, y1, x2, y2

speaker_box = normalized_roi_to_box(
    roi_data["speaker_roi"],
    width,
    height,
)

dialogue_box = normalized_roi_to_box(
    roi_data["dialogue_roi"],
    width,
    height,
)

preview = frame.copy()
cv2.rectangle(preview, speaker_box[:2], speaker_box[2:], (0, 255, 0), 3)
cv2.rectangle(preview, dialogue_box[:2], dialogue_box[2:], (255, 0, 0), 3)

print(f"{width}x{height}, fps={fps:.3f}, duration={duration:.1f}s")
print(json.dumps(roi_data, ensure_ascii=False, indent=2))

plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(preview, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()


In [ ]:
SAMPLE_INTERVAL_SEC = 0.5 # @param {type:"number"}
PREVIEW_EVERY_N = 100 # @param {type:"integer"}

from collections import defaultdict
from difflib import SequenceMatcher
from IPython.display import clear_output, display
from tqdm.auto import tqdm
from urllib.request import urlopen
import math
import pandas as pd

CHARACTER_NAMES_URL = "https://raw.githubusercontent.com/HisameOgasahara/Data_crawling/main/hbr_youtube_ocr/character_names.txt"

with urlopen(CHARACTER_NAMES_URL) as response:
    known_names = [
        line.strip()
        for line in response.read().decode("utf-8").splitlines()
        if line.strip()
    ]

roi_data = json.loads(roi_json_path.read_text(encoding="utf-8"))

cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = cap.get(cv2.CAP_PROP_FRAME_COUNT) / fps

speaker_box = normalized_roi_to_box(roi_data["speaker_roi"], width, height)
dialogue_box = normalized_roi_to_box(roi_data["dialogue_roi"], width, height)

def similar(a, b):
    if not a or not b:
        return False
    return a.startswith(b) or b.startswith(a) or SequenceMatcher(None, a, b).ratio() >= 0.82

def match_known_name(raw_name):
    raw_name = norm(raw_name)
    if not raw_name:
        return "", 0.0

    best_name = raw_name
    best_score = 0.0

    for candidate in known_names:
        candidate_norm = norm(candidate)

        if raw_name == candidate_norm:
            return candidate, 1.0

        score = SequenceMatcher(None, raw_name, candidate_norm).ratio()

        if raw_name in candidate_norm or candidate_norm in raw_name:
            score = max(score, 0.85)

        if score > best_score:
            best_name = candidate
            best_score = score

    if best_score < 0.55:
        return "", best_score

    return best_name, best_score

total_samples = math.floor(duration / SAMPLE_INTERVAL_SEC) + 1
rows = []
progress = tqdm(total=total_samples, desc="OCR", unit="sample")

for sample_index in range(total_samples):
    t = sample_index * SAMPLE_INTERVAL_SEC
    cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
    ok, img = cap.read()
    if not ok:
        break

    speaker_raw = norm(mocr(prep(crop_roi(img, speaker_box))))
    speaker, speaker_score = match_known_name(speaker_raw)
    dialogue = norm(mocr(prep(crop_roi(img, dialogue_box))))

    rows.append({
        "time_sec": round(t, 3),
        "speaker_raw": speaker_raw,
        "speaker": speaker,
        "speaker_score": round(speaker_score, 4),
        "dialogue": dialogue,
    })

    progress.update(1)
    progress.set_postfix({
        "time": f"{t:.1f}/{duration:.1f}s",
        "speaker": speaker[:12],
    })

    if PREVIEW_EVERY_N > 0 and (sample_index % PREVIEW_EVERY_N == 0 or sample_index == total_samples - 1):
        clear_output(wait=True)
        display(progress.container)
        print(f"sample: {sample_index + 1}/{total_samples}")
        print(f"time: {t:.1f}/{duration:.1f} sec")
        print("speaker_raw:", speaker_raw)
        print("speaker:", speaker, f"({speaker_score:.3f})")
        print("dialogue:", dialogue)

        debug = img.copy()
        cv2.rectangle(debug, speaker_box[:2], speaker_box[2:], (0, 255, 0), 3)
        cv2.rectangle(debug, dialogue_box[:2], dialogue_box[2:], (255, 0, 0), 3)

        plt.figure(figsize=(14, 8))
        plt.imshow(cv2.cvtColor(debug, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.show()

cap.release()
progress.close()

raw_df = pd.DataFrame(rows)
raw_df.to_csv(output_dir / "ocr_samples.csv", index=False, encoding="utf-8-sig")

segments = []
current = None

for row in rows:
    s = row["speaker"]
    d = row["dialogue"]
    ts = row["time_sec"]

    if not s or not d:
        if current is not None:
            current["end_sec"] = ts
            segments.append(current)
            current = None
        continue

    if current is None:
        current = {
            "start_sec": ts,
            "end_sec": ts + SAMPLE_INTERVAL_SEC,
            "speaker": s,
            "dialogue": d,
        }
        continue

    if s == current["speaker"] and similar(d, current["dialogue"]):
        current["end_sec"] = ts + SAMPLE_INTERVAL_SEC
        if len(d) > len(current["dialogue"]):
            current["dialogue"] = d
        continue

    segments.append(current)
    current = {
        "start_sec": ts,
        "end_sec": ts + SAMPLE_INTERVAL_SEC,
        "speaker": s,
        "dialogue": d,
    }

if current is not None:
    segments.append(current)

segments_df = pd.DataFrame(segments)
segments_df.to_csv(output_dir / "dialogue_segments.csv", index=False, encoding="utf-8-sig")

character_dir = output_dir / "characters"
character_dir.mkdir(parents=True, exist_ok=True)

by_character = defaultdict(list)
for segment in segments:
    by_character[segment["speaker"]].append(segment)

def format_timestamp(seconds):
    total_ms = int(round(seconds * 1000))
    minutes = total_ms // 60000
    remaining_ms = total_ms % 60000
    whole_seconds = remaining_ms // 1000
    milliseconds = remaining_ms % 1000
    return f"{minutes:02d}:{whole_seconds:02d}.{milliseconds:03d}"

for speaker, speaker_segments in by_character.items():
    lines = []

    for segment in speaker_segments:
        start = format_timestamp(segment["start_sec"])
        end = format_timestamp(segment["end_sec"])
        lines.append(f"[{start} - {end}]")
        lines.append(segment["dialogue"])
        lines.append("")

    (character_dir / f"{speaker}.txt").write_text(
        "\n".join(lines),
        encoding="utf-8",
    )

print(f"done: {len(rows)}/{total_samples} samples")
print("saved:", output_dir / "ocr_samples.csv")
print("saved:", output_dir / "dialogue_segments.csv")
print("saved character TXT files:", character_dir)
display(segments_df.head(50))


In [ ]:
SAMPLE_INTERVAL_SEC = 0.5 # @param {type:"number"}
PREVIEW_EVERY_N = 100 # @param {type:"integer"}

from difflib import SequenceMatcher
from IPython.display import clear_output, display
from tqdm.auto import tqdm
import math
import pandas as pd

def similar(a, b):
    if not a or not b:
        return False
    return a.startswith(b) or b.startswith(a) or SequenceMatcher(None,a,b).ratio() >= 0.82

cap = cv2.VideoCapture(str(video_path))
duration = cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS)
total_samples = math.floor(duration / SAMPLE_INTERVAL_SEC) + 1
rows = []

progress = tqdm(total=total_samples, desc="OCR", unit="sample")

for sample_index in range(total_samples):
    t = sample_index * SAMPLE_INTERVAL_SEC
    cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
    ok, img = cap.read()
    if not ok:
        break

    speaker = norm(mocr(prep(crop_roi(img, speaker_box))))
    dialogue = norm(mocr(prep(crop_roi(img, dialogue_box))))

    rows.append({
        "time_sec": round(t, 3),
        "speaker": speaker,
        "dialogue": dialogue,
    })

    progress.update(1)
    progress.set_postfix({
        "time": f"{t:.1f}/{duration:.1f}s",
        "speaker": speaker[:12],
    })

    if PREVIEW_EVERY_N > 0 and (sample_index % PREVIEW_EVERY_N == 0 or sample_index == total_samples - 1):
        clear_output(wait=True)
        display(progress.container)
        print(f"sample: {sample_index + 1}/{total_samples}")
        print(f"time: {t:.1f}/{duration:.1f} sec")
        print("speaker:", speaker)
        print("dialogue:", dialogue)

        debug = img.copy()
        cv2.rectangle(debug, speaker_box[:2], speaker_box[2:], (0, 255, 0), 3)
        cv2.rectangle(debug, dialogue_box[:2], dialogue_box[2:], (255, 0, 0), 3)

        plt.figure(figsize=(14, 8))
        plt.imshow(cv2.cvtColor(debug, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.show()

cap.release()
progress.close()

raw_df = pd.DataFrame(rows)
raw_df.to_csv(output_dir / "ocr_raw_samples.csv", index=False, encoding="utf-8-sig")

segments = []
current = None
for row in rows:
    s, d, ts = row["speaker"], row["dialogue"], row["time_sec"]
    if not d:
        if current is not None:
            current["end_sec"] = ts
            segments.append(current)
            current = None
        continue

    if current is None:
        current = {
            "start_sec": ts,
            "end_sec": ts,
            "speaker": s,
            "dialogue": d,
        }
        continue

    if (s == current["speaker"] or not s) and similar(d, current["dialogue"]):
        current["end_sec"] = ts
        if len(d) > len(current["dialogue"]):
            current["dialogue"] = d
        if not current["speaker"] and s:
            current["speaker"] = s
    else:
        segments.append(current)
        current = {
            "start_sec": ts,
            "end_sec": ts,
            "speaker": s,
            "dialogue": d,
        }

if current is not None:
    segments.append(current)

segments_df = pd.DataFrame(segments)
segments_df.to_csv(output_dir / "dialogue_segments.csv", index=False, encoding="utf-8-sig")

print(f"done: {len(rows)}/{total_samples} samples")
display(segments_df.head(50))
